### IMPORTS

In [1]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import FrenchStemmer
from collections import Counter

nltk.download('stopwords', quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec, FastText

# Style global
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

from sentence_transformers import SentenceTransformer


### LOAD DATA

In [2]:
df_clean = pd.read_csv("../data/df_clean.csv")

text_col = "Rapport_Collecte"
target   = "Categorie"

X = df_clean[text_col].astype(str)
y = df_clean[target]

print(f"Dimensions : {df_clean.shape[0]} lignes x {df_clean.shape[1]} colonnes")

Dimensions : 9724 lignes x 9 colonnes


### SPLIT

In [3]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

print(X_train.shape, X_val.shape, X_test.shape)

(6806,) (1459,) (1459,)


### PRÉTRAITEMENT TEXTUEL

Pipeline : normalisation → tokenisation → suppression des stopwords (français + domaine) → stemming (FrenchStemmer)

In [4]:
# ─── STOPWORDS FRANÇAIS (NLTK) ───────────────────────────────────────────────
STOPWORDS_FR = set(stopwords.words("french"))


STOPWORDS_DOMAINE = {

    # administratif
    "rapport","collecte","observation","note","constat",
    "visite","lot","provenance","provenant","issu","origine",

    # lieux
    "site","zone","secteur","centre","usine","municipal",
    "localisation","emplacement",

    # organisation
    "agent","équipe","general","général",

    # qualité
    "standard","standards","conforme",
    "verification","vérification",
    "recommandee","recommandée",
    "prioritaire",

    # mesures déjà présentes en colonnes numériques
    "poids","volume","masse","kg","l",
    "mesuré","mesure","estimé","estime",

    # intensité inutile
    "très","extremement","extrêmement",
    "moyen","moyenne","faible","élevée",

    # mots vides dataset
    "non","aucune","aucun","sans"
}

STOPWORDS_TOUS = STOPWORDS_FR | STOPWORDS_DOMAINE

stemmer = FrenchStemmer()


def preprocess(text: str) -> str:
    """
    Pipeline NLP :
    1. Mise en minuscules
    2. Suppression des caractères non-alphabétiques (garde accents français)
    3. Tokenisation par split
    4. Suppression des stopwords FR + domaine (version corrigée)
    5. Suppression des tokens ≤ 2 caractères
    6. Stemming FrenchStemmer (NLTK Snowball)
    Retourne une chaîne de tokens traités rejoints par espaces.
    """
    text = text.lower()
    text = re.sub(r"[^a-zàâäéèêëîïôùûüç\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS_TOUS and len(t) > 2]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)


# Application sur les trois partitions
X_train_clean = X_train.apply(preprocess)
X_val_clean   = X_val.apply(preprocess)
X_test_clean  = X_test.apply(preprocess)

# Vérification rapide
print("Avant :", X_train.iloc[0])
print("Après :", X_train_clean.iloc[0])
print(f"\nVocabulaire train : {len(set(' '.join(X_train_clean).split()))} tokens uniques")

Avant : Lot de verre trié en provenance de l'Usine A. Masse 185.6 kg. Matériau lourd et extrêmement rigide. Aspect indéterminée. Étiquetage absent sur le conteneur.
Après : verr tri matériau lourd rigid aspect indétermin étiquetag absent conteneur

Vocabulaire train : 116 tokens uniques


### MODELES

In [5]:
# Classificateurs compatibles avec les représentations creuses (BoW / TF-IDF)
models_text = {
    "Naive Bayes"         : MultinomialNB(),
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Linear SVC"          : LinearSVC(),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42)
}

# Classificateurs compatibles avec les embeddings denses (Word2Vec / FastText / CamemBERT)
# MultinomialNB est exclu : il exige des features non-négatives
models_emb = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Linear SVC"          : LinearSVC(),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42)
}

### FONCTION EVALUATION

In [6]:
def evaluate(model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)

    acc  = accuracy_score(yte, pred)
    prec = precision_score(yte, pred, average="weighted")
    rec  = recall_score(yte, pred, average="weighted")
    f1   = f1_score(yte, pred, average="weighted")

    return acc, prec, rec, f1

### BAG OF WORDS (BASELINE)

In [7]:
print("\n=== BAG OF WORDS ===")

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train_clean)
X_test_bow  = bow.transform(X_test_clean)

results_bow = {}

for name, model in models_text.items():
    acc, prec, rec, f1 = evaluate(model, X_train_bow, X_test_bow, y_train, y_test)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results_bow[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}


=== BAG OF WORDS ===

=== Naive Bayes ===
Accuracy : 0.9650
Precision: 0.9666
Recall   : 0.9650
F1-score : 0.9653

=== Logistic Regression ===
Accuracy : 0.9623
Precision: 0.9635
Recall   : 0.9623
F1-score : 0.9625

=== Linear SVC ===
Accuracy : 0.9623
Precision: 0.9635
Recall   : 0.9623
F1-score : 0.9625

=== Random Forest ===
Accuracy : 0.9644
Precision: 0.9647
Recall   : 0.9644
F1-score : 0.9644


### TF-IDF (REFERENCE)

In [8]:
print("\n=== TF-IDF ===")

tfidf = TfidfVectorizer(ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train_clean)
X_test_tfidf  = tfidf.transform(X_test_clean)

results_tfidf = {}

for name, model in models_text.items():
    acc, prec, rec, f1 = evaluate(model, X_train_tfidf, X_test_tfidf, y_train, y_test)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results_tfidf[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}


=== TF-IDF ===

=== Naive Bayes ===
Accuracy : 0.9616
Precision: 0.9634
Recall   : 0.9616
F1-score : 0.9619

=== Logistic Regression ===
Accuracy : 0.9630
Precision: 0.9656
Recall   : 0.9630
F1-score : 0.9635

=== Linear SVC ===
Accuracy : 0.9637
Precision: 0.9644
Recall   : 0.9637
F1-score : 0.9638

=== Random Forest ===
Accuracy : 0.9650
Precision: 0.9655
Recall   : 0.9650
F1-score : 0.9651


### WORD2VEC (SÉMANTIQUE)

In [9]:
# Tokenisation sur les textes prétraités
sentences = [text.split() for text in X_train_clean]

w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)


def vectorize_w2v(text: str) -> np.ndarray:
    words = text.split()
    vecs  = [w2v.wv[w] for w in words if w in w2v.wv]
    if not vecs:
        return np.zeros(100)
    return np.mean(vecs, axis=0)


X_train_w2v = np.vstack([vectorize_w2v(t) for t in X_train_clean]).astype(np.float32)
X_test_w2v  = np.vstack([vectorize_w2v(t) for t in X_test_clean]).astype(np.float32)

results_w2v = {}

for name, model in models_emb.items():
    acc, prec, rec, f1 = evaluate(model, X_train_w2v, X_test_w2v, y_train, y_test)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results_w2v[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}


=== Logistic Regression ===
Accuracy : 0.9623
Precision: 0.9638
Recall   : 0.9623
F1-score : 0.9626

=== Linear SVC ===
Accuracy : 0.9609
Precision: 0.9633
Recall   : 0.9609
F1-score : 0.9613

=== Random Forest ===
Accuracy : 0.9664
Precision: 0.9668
Recall   : 0.9664
F1-score : 0.9665


### FASTTEXT

FastText génère des vecteurs pour **tous** les mots y compris les mots hors-vocabulaire grâce aux n-grammes de caractères. Le filtre `if w in ft.wv` est donc intentionnellement omis ici.

In [10]:
ft = FastText(sentences, vector_size=100, window=5, min_count=2, workers=4)


def vectorize_ft(text: str) -> np.ndarray:
    words = text.split()
    if not words:
        return np.zeros(100)
    # ft.wv[w] fonctionne même pour les mots OOV (hors vocabulaire)
    # grâce aux n-grammes de caractères — ne pas filtrer avec 'if w in ft.wv'
    vecs = [ft.wv[w] for w in words]
    return np.mean(vecs, axis=0)


X_train_ft = np.vstack([vectorize_ft(t) for t in X_train_clean]).astype(np.float32)
X_test_ft  = np.vstack([vectorize_ft(t) for t in X_test_clean]).astype(np.float32)

results_fasttext = {}

for name, model in models_emb.items():
    acc, prec, rec, f1 = evaluate(model, X_train_ft, X_test_ft, y_train, y_test)

    print(f"\n=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    results_fasttext[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}


=== Logistic Regression ===
Accuracy : 0.9637
Precision: 0.9642
Recall   : 0.9637
F1-score : 0.9638

=== Linear SVC ===
Accuracy : 0.9637
Precision: 0.9651
Recall   : 0.9637
F1-score : 0.9639

=== Random Forest ===
Accuracy : 0.9657
Precision: 0.9660
Recall   : 0.9657
F1-score : 0.9658
